# 实验二 D：基于 CNN 的文本情感分类

**实验目的**

1. 基于 IMDB Movie Review Dataset，完成影评情感分类（正向 / 负向）任务；
2. 掌握文本数据的向量化表示（词表 + 词嵌入）与 Text-CNN 的特征提取方式；
3. 建立端到端的文本分类模型，通过损失下降曲线与测试集性能，理解 CNN 在序列数据上的局部特征捕捉能力。

**数据**：`imdb/` 目录下的 `train.jsonl.gz` 与 `test.jsonl.gz`，每行是一条影评
（`{"text": 评论文本, "label": 0 负向 / 1 正向}`），训练集与测试集各 25000 条、正负各半。
该数据即 [ModelScope 的 imdb 数据集](https://www.modelscope.cn/datasets/modelscope/imdb/summary) 的内容
（源自 Stanford 的 IMDB 影评数据），已预先转成便于读取的 gzip JSONL 格式。
为控制 CPU 上的训练时间，本实验只取其中一部分样本，改一行常量即可用全部数据。

In [1]:
import gzip
import json
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader, Dataset

pio.templates.default = "plotly_white"
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

C_BLUE, C_ORANGE, C_RED = "#2a78d6", "#eb6834", "#e34948"
C_INK, C_MUTED = "#0b0b0b", "#52514e"
BLUES = [[0.0, "#f2f7fe"], [0.35, "#cde2fb"], [0.7, "#6da7ec"], [1.0, "#2a78d6"]]

DATA_DIR = Path("imdb")          # train.jsonl.gz / test.jsonl.gz
print("torch", torch.__version__, "| 设备：cpu" if not torch.cuda.is_available() else "| 设备：cuda")

torch 2.13.0 | 设备：cpu


In [2]:
def load_split(name):
    """读取 gzip 压缩的 jsonl，每行 {"text": ..., "label": 0/1}"""
    with gzip.open(DATA_DIR / f"{name}.jsonl.gz", "rt", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


train_all, test_all = load_split("train"), load_split("test")
print(f"训练集 {len(train_all)} 条，测试集 {len(test_all)} 条（正负各半）")
print("标签含义：0 = 负面，1 = 正面\n")

sample = train_all[0]
print(f"一条影评的前 180 个字符：\n{sample['text'][:180]}...")
print(f"真实标签：{sample['label']}（{'正向' if sample['label'] == 1 else '负向'}）")

lens = [len(r["text"].split()) for r in train_all]
print(f"\n影评长度（按空格切分）：平均 {np.mean(lens):.0f} 词，中位数 {np.median(lens):.0f} 词，最长 {max(lens)} 词")

训练集 25000 条，测试集 25000 条（正负各半）
标签含义：0 = 负面，1 = 正面

一条影评的前 180 个字符：
Eric Clapton, Jack Bruce and Ginger Baker re-unite to play all their songs from 35 years ago when they formed a trio called "Cream." Those were the psychedelic days of England and ...
真实标签：1（正向）

影评长度（按空格切分）：平均 234 词，中位数 174 词，最长 2470 词


## 一、文本向量化

神经网络无法直接处理字符串，需要先把文本变成数字序列，分三步：

1. **分词**：把句子切成词。这里用最简单的方式——转小写后用正则 `[a-z0-9']+` 提取词元，
   标点被自然滤掉，也不需要额外的分词工具；
2. **建词表**：统计训练集里的词频，给每个词一个整数编号，另加两个特殊符号
   `<pad>`（补齐用）和 `<unk>`（未登录词）；
3. **编码**：把每条评论转成等长的编号序列——太长的截断到固定长度，太短的用 `<pad>` 补到同样长度，
   这样才能组成批次送进网络。

词嵌入（Embedding）层则把每个编号映射成一个稠密向量，这些向量**随网络一起训练**，
语义相近的词会在训练中逐渐靠近，这是让 CNN 能处理文本的关键一步。

In [3]:
TOKEN_RE = re.compile(r"[a-z0-9']+")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())


MAX_LEN = 200        # 每条评论固定截断/补齐到 200 个词
MIN_FREQ = 5         # 训练集中出现次数少于 5 的词统一当作 <unk>
N_TRAIN, N_TEST = 10000, 3000   # 用一部分数据训练，控制 CPU 上的耗时

train_data, test_data = train_all[:N_TRAIN], test_all[:N_TEST]
# 数据文件已打乱过顺序，这里再确认一次每个子集都是正负均衡的
for name, rows in (("训练子集", train_data), ("测试子集", test_data)):
    pos = np.mean([r["label"] for r in rows])
    assert 0.4 < pos < 0.6, f"{name}的标签分布异常：正向占 {pos:.1%}"
    print(f"{name}：{len(rows)} 条，正向占 {pos:.1%}")

counter = Counter()
for row in train_data:
    counter.update(tokenize(row["text"]))

vocab = {"<pad>": 0, "<unk>": 1}
for word, freq in counter.most_common():
    if freq < MIN_FREQ:
        break
    vocab[word] = len(vocab)

print(f"训练集共出现 {len(counter):,} 个不同的词，保留词表大小 {len(vocab):,}")
print(f"词表覆盖了训练集全部词频的 {sum(c for _, c in counter.most_common(len(vocab) - 2)) / sum(counter.values()):.1%}")
print("词表前 12 个词：", list(vocab)[:12])


def encode(text):
    """转小写 -> 分词 -> 查词表（未登录词用 <unk>）-> 截断/补齐到 MAX_LEN"""
    ids = [vocab.get(w, 1) for w in tokenize(text)][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))


tokens = tokenize(train_data[0]["text"])[:12]
print(f"\n示例：{tokens}\n编码：{encode(train_data[0]['text'])[:12]}")
truncated = np.mean([len(tokenize(r["text"])) > MAX_LEN for r in train_data])
print(f"长度超过 {MAX_LEN} 词、被截断的评论占比：{truncated:.1%}（被截掉的是评论结尾，对情感判断影响不大）")

训练子集：10000 条，正向占 50.1%
测试子集：3000 条，正向占 49.7%


训练集共出现 57,845 个不同的词，保留词表大小 18,867
词表覆盖了训练集全部词频的 97.3%
词表前 12 个词： ['<pad>', '<unk>', 'the', 'and', 'a', 'of', 'to', 'is', 'br', 'in', 'it', 'i']

示例：['eric', 'clapton', 'jack', 'bruce', 'and', 'ginger', 'baker', 're', 'unite', 'to', 'play', 'all']
编码：[1909, 1, 710, 1525, 3, 5390, 2868, 783, 12759, 6, 309, 30]
长度超过 200 词、被截断的评论占比：42.1%（被截掉的是评论结尾，对情感判断影响不大）


## 二、Text-CNN 结构

Text-CNN（Kim, 2014）把一维卷积用在词向量序列上：卷积核在**时间维**上滑动，
一次只看连续 k 个词（k = 3、4、5），因此捕捉的是“若干个相邻词组成的局部短语”，
例如 `not very good`、`waste of time` 这类决定情感倾向的片段。

| 层 | 结构 | 说明 |
|---|---|---|
| 词嵌入 | Embedding(词表, 64) | 把编号变成 64 维向量，随网络训练 |
| 卷积 | 3 组 Conv1d(64→64)，核长度 3/4/5 | 每组对应一种 n-gram 视野，输出 64 张特征图 |
| 池化 | 每组在时间维上做**全局最大池化** | 取“最强烈的那处局部特征”，因此与句子长度无关 |
| 分类 | Dropout(0.3) → Linear(192→2) | 三组池化结果拼接后输出正负两类得分 |

与图像 CNN 的区别在于：图像卷积核在二维平面上滑动，文本卷积核只沿序列方向滑动——
但“局部感受野 + 权值共享 + 池化”的思路完全一致。

In [4]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, n_filters=64,
                 kernel_sizes=(3, 4, 5), n_classes=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, n_filters, k) for k in kernel_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(n_filters * len(kernel_sizes), n_classes)

    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)              # (B, L) -> (B, emb, L)
        pooled = [torch.relu(conv(emb)).max(dim=2).values    # 每个卷积核取时间维最大值
                  for conv in self.convs]
        return self.fc(self.dropout(torch.cat(pooled, dim=1)))


model = TextCNN(len(vocab))
print(f"模型参数量：{sum(p.numel() for p in model.parameters()):,}"
      f"（其中词嵌入 {model.embedding.weight.numel():,}）")
print(model)

模型参数量：1,257,218（其中词嵌入 1,207,488）
TextCNN(
  (embedding): Embedding(18867, 64, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(64, 64, kernel_size=(3,), stride=(1,))
    (1): Conv1d(64, 64, kernel_size=(4,), stride=(1,))
    (2): Conv1d(64, 64, kernel_size=(5,), stride=(1,))
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=192, out_features=2, bias=True)
)


## 三、训练过程

损失函数用交叉熵，优化器用 Adam（学习率 1e-3），训练 5 轮，批量大小 64。

In [5]:
class ReviewDataset(Dataset):
    def __init__(self, rows):
        self.X = torch.tensor([encode(r["text"]) for r in rows], dtype=torch.long)
        self.y = torch.tensor([r["label"] for r in rows], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


train_ds, test_ds = ReviewDataset(train_data), ReviewDataset(test_data)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)
print(f"训练集 {len(train_ds)} 条，测试集 {len(test_ds)} 条")

训练集 10000 条，测试集 3000 条


In [6]:
EPOCHS, LR = 5, 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def run_epoch(loader, train=False):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in loader:
        with torch.set_grad_enabled(train):
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * y_batch.size(0)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += y_batch.size(0)
    return total_loss / total, correct / total


history = {"epoch": [], "train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}
t0 = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    te_loss, te_acc = run_epoch(test_loader)
    history["epoch"].append(epoch)
    for key, value in [("train_loss", tr_loss), ("test_loss", te_loss),
                       ("train_acc", tr_acc), ("test_acc", te_acc)]:
        history[key].append(value)
    print(f"epoch {epoch} | 训练损失 {tr_loss:.4f} 准确率 {tr_acc:.4f} | "
          f"测试损失 {te_loss:.4f} 准确率 {te_acc:.4f}")
print(f"\n训练 {EPOCHS} 轮共耗时 {time.perf_counter() - t0:.0f} 秒")

epoch 1 | 训练损失 0.6977 准确率 0.5811 | 测试损失 0.6006 准确率 0.6457


epoch 2 | 训练损失 0.5641 准确率 0.7048 | 测试损失 0.5063 准确率 0.7713


epoch 3 | 训练损失 0.4886 准确率 0.7579 | 测试损失 0.4641 准确率 0.7837


epoch 4 | 训练损失 0.4298 准确率 0.8004 | 测试损失 0.4281 准确率 0.8027


epoch 5 | 训练损失 0.3748 准确率 0.8342 | 测试损失 0.4029 准确率 0.8170

训练 5 轮共耗时 120 秒


In [7]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=("损失曲线", "准确率曲线"))
for key, name, color, col in [("train_loss", "训练损失", C_BLUE, 1), ("test_loss", "测试损失", C_ORANGE, 1),
                              ("train_acc", "训练准确率", C_BLUE, 2), ("test_acc", "测试准确率", C_ORANGE, 2)]:
    fig.add_trace(go.Scatter(
        x=history["epoch"], y=history[key], mode="lines+markers", name=name,
        line=dict(color=color, width=2), marker=dict(size=7),
        hovertemplate=f"{name}<br>epoch %{{x}}：%{{y:.4f}}<extra></extra>",
    ), row=1, col=col)

fig.update_xaxes(title_text="epoch", dtick=1, row=1, col=1)
fig.update_xaxes(title_text="epoch", dtick=1, row=1, col=2)
fig.update_yaxes(title_text="交叉熵损失", row=1, col=1)
fig.update_yaxes(title_text="准确率", range=[0.5, 1.0], tickformat=".0%", row=1, col=2)
fig.update_layout(title=dict(text="训练过程：损失持续下降，测试准确率稳步上升", x=0.02),
                  width=1080, height=440, margin=dict(l=70, r=30, t=90, b=80),
                  legend=dict(orientation="h", yanchor="top", y=-0.2, x=0, title=None))
fig.show()

## 四、测试集性能分析

In [8]:
model.eval()
y_true, y_pred, y_score = [], [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        y_pred += logits.argmax(1).tolist()
        y_true += y_batch.tolist()
        y_score += torch.softmax(logits, dim=1)[:, 1].tolist()

acc = float(np.mean(np.array(y_true) == np.array(y_pred)))
cm = confusion_matrix(y_true, y_pred)
print(f"测试集准确率：{acc:.4f}（{len(y_true)} 条中判对 {int(acc * len(y_true))} 条）")
print(f"混淆矩阵（行=真实，列=预测）：\n{cm}")

fig = go.Figure(go.Heatmap(
    z=cm[::-1], x=["预测 负向", "预测 正向"], y=["真实 正向", "真实 负向"],
    colorscale=BLUES, zmin=0, zmax=cm.max(), showscale=False,
    text=cm[::-1], texttemplate="%{text}", textfont=dict(size=16, color=C_INK),
    hovertemplate="%{y} → %{x}：%{z} 条<extra></extra>",
))
fig.update_layout(title=dict(text=f"测试集混淆矩阵（准确率 {acc:.1%}）", x=0.02),
                  width=520, height=470, margin=dict(l=90, r=30, t=70, b=70))
fig.show()

测试集准确率：0.8170（3000 条中判对 2451 条）
混淆矩阵（行=真实，列=预测）：
[[1266  243]
 [ 306 1185]]


In [9]:
# 看几条具体例子：判对与判错各取 3 条（按预测为正的概率排序，展示最有把握和最没把握的）
df_pred = pd.DataFrame({
    "文本片段": [r["text"][:130].replace("\n", " ") + "……" for r in test_data],
    "真实情感": ["正向" if r["label"] == 1 else "负向" for r in test_data],
    "预测情感": ["正向" if p == 1 else "负向" for p in y_pred],
    "预测为正的概率": np.round(y_score, 3),
})
df_pred["是否正确"] = np.where(df_pred["真实情感"] == df_pred["预测情感"], "✓", "✗")

# 判对的取模型最有把握的两个方向各 2 条；判错的取模型“错得最有把握”的 3 条
sure_positive = df_pred[(df_pred["是否正确"] == "✓") & (df_pred["预测情感"] == "正向")] \
    .nlargest(2, "预测为正的概率")
sure_negative = df_pred[(df_pred["是否正确"] == "✓") & (df_pred["预测情感"] == "负向")] \
    .nsmallest(2, "预测为正的概率")
wrong = df_pred[df_pred["是否正确"] == "✗"].copy()
wrong["把握"] = (wrong["预测为正的概率"] - 0.5).abs()
confident_wrong = wrong.nlargest(3, "把握").drop(columns="把握")

examples = pd.concat([
    sure_positive.assign(情形="判对：强烈判为正向"),
    sure_negative.assign(情形="判对：强烈判为负向"),
    confident_wrong.assign(情形="判错：模型却很有把握"),
])[["情形", "文本片段", "真实情感", "预测情感", "预测为正的概率"]]
examples

,情形,文本片段,真实情感,预测情感,预测为正的概率
1389,判对：强烈判为正向,An absoloutely wonderful film that works on se...,正向,正向,0.999
2533,判对：强烈判为正向,This Movie was Great and Funny. Pauly is Funny...,正向,正向,0.998
926,判对：强烈判为负向,This movie was terrible. You couldn't fast for...,负向,负向,0.000
1566,判对：强烈判为负向,"Without doubt, this is the worst movie I've ev...",负向,负向,0.000
1733,判错：模型却很有把握,"Okay, I'll say it. This movie made me laugh so...",正向,负向,0.008
190,判错：模型却很有把握,"Okay, I think we're all agreed that Michael Ja...",正向,负向,0.009
2392,判错：模型却很有把握,The movie was very good. I'm an avid mystery f...,正向,负向,0.027


从判对的例子可以看到模型的判断方式与人的直觉一致：出现 `wonderful`、`great`、`funny` 的评论被强烈判为正向
（概率 0.998 以上），出现 `terrible`、`worst` 的评论被强烈判为负向（概率 0.000）——
这正是 3/4/5 元卷积核在局部窗口上捕捉到的情感短语。

更有意思的是三条**判错**的样本，它们能说明 Text-CNN 的局限在哪里：

* 一条真实为正向的影评开头写着 `The movie was very good`，但后文在评价演员阵容时出现了
  `The worst was Peter Ustinov!`。这里的 `worst` 其实只针对某位演员，全局最大池化却把它当作全篇最强的
  局部证据，于是整条评论被判成负向——**词的情感极性依赖上下文，而最大池化会丢掉这个上下文**。
* 另一条是在夸一部“烂片”：`this movie was intended to be bad and cheezy`、
  `corny dialogue, bad one liners and horrible special effects`，作者其实是抱着欣赏恶搞的态度在赞美它。
  全篇都是负面词汇，模型判为负向（概率 0.008），而人类读者能读出反讽——**这类反讽与特殊语体，
  仅靠词级局部特征无法判别**。

从混淆矩阵也能看出同一现象：负向评论判对 1266/1509（83.9%），正向评论判对 1185/1491（79.5%），
正向略低，正是因为上述“正向评论里出现负面短语”的情况更多。这说明 Text-CNN 学到的证据停留在
**关键短语**这一层，缺少对句法结构、否定与整篇语境的建模能力，这也正是后来循环网络与注意力机制出现的原因。

## 五、实验小结

本实验用「词表 + 词嵌入 + 三组不同窗口的一维卷积 + 全局最大池化」搭出了一个端到端的文本分类模型，
在 10000 条训练样本上训练 5 轮后，测试集准确率达到 **81.7%**，损失从 0.60 稳定下降到 0.40，
训练与测试曲线始终贴合、没有出现明显的过拟合。

需要注意的是，第 5 轮时测试准确率**仍在缓慢上升**（第 4 轮到第 5 轮从 80.3% 升到 81.7%），
说明模型还没有训练充分；在 CPU 上增加训练轮数、或者把 `N_TRAIN` 调大用上全部 25000 条训练数据，
准确率都还能继续提高（这类模型用全部数据训练通常在 86%–88%）。
另外，本实验只用了随机初始化的词嵌入，若换成 GloVe 等预训练词向量，小样本下还能有明显提升。

In [10]:
print("=" * 62)
print("实验结论（关键数值）")
print("=" * 62)
print(f"数据：IMDB 影评，训练 {len(train_ds)} 条 / 测试 {len(test_ds)} 条（正负各半），"
      f"原文长度平均 {np.mean(lens):.0f} 词")
print(f"向量化：词表 {len(vocab):,} 个词（词频 >= {MIN_FREQ}），每条截断/补齐到 {MAX_LEN} 个词，"
      f"词嵌入维度 64 且随训练更新")
print(f"模型：Embedding + 3 组 Conv1d（核长 3/4/5，各 64 个特征图）+ 全局最大池化 + 全连接，"
      f"参数量 {sum(p.numel() for p in model.parameters()):,}")
print("-" * 62)
print(f"训练 {EPOCHS} 轮：训练损失 {history['train_loss'][0]:.4f} -> {history['train_loss'][-1]:.4f}，"
      f"测试损失 {history['test_loss'][0]:.4f} -> {history['test_loss'][-1]:.4f}")
print(f"准确率：训练 {history['train_acc'][0]:.4f} -> {history['train_acc'][-1]:.4f}，"
      f"测试 {history['test_acc'][0]:.4f} -> {history['test_acc'][-1]:.4f}")
print(f"测试集最终准确率 {acc:.4f}；基准线：随机猜测 0.5000")
print(f"混淆矩阵：负向判对 {cm[0, 0]} / {cm[0].sum()}，正向判对 {cm[1, 1]} / {cm[1].sum()}")
print("=" * 62)

实验结论（关键数值）
数据：IMDB 影评，训练 10000 条 / 测试 3000 条（正负各半），原文长度平均 234 词
向量化：词表 18,867 个词（词频 >= 5），每条截断/补齐到 200 个词，词嵌入维度 64 且随训练更新
模型：Embedding + 3 组 Conv1d（核长 3/4/5，各 64 个特征图）+ 全局最大池化 + 全连接，参数量 1,257,218
--------------------------------------------------------------
训练 5 轮：训练损失 0.6977 -> 0.3748，测试损失 0.6006 -> 0.4029
准确率：训练 0.5811 -> 0.8342，测试 0.6457 -> 0.8170
测试集最终准确率 0.8170；基准线：随机猜测 0.5000
混淆矩阵：负向判对 1266 / 1509，正向判对 1185 / 1491
